In [0]:
from pyspark.sql.functions import col, trim, lower, to_timestamp, datediff, year, month, when, sum as _sum

In [0]:
df = spark.table("ecommerce_dev.bronze.orders_raw")

orders_enhanced = df \
    .dropDuplicates(["order_id"]) \
    .filter(col("order_id").isNotNull()) \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")) \
    .filter(
        col("order_delivered_customer_date").isNull() |
        (col("order_delivered_customer_date") >= col("order_purchase_timestamp"))
    ) \
    .withColumn("delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ) \
    .withColumn("order_year", year(col("order_purchase_timestamp"))) \
    .withColumn("order_month", month(col("order_purchase_timestamp"))) \
    .withColumn("is_delivered", col("order_status") == "delivered") \
    .withColumn(
        "is_late_delivery",
        when(
            col("order_delivered_customer_date").isNull() |
            (col("order_delivered_customer_date") > col("order_estimated_delivery_date")),
            True
        ).otherwise(False)
    )

    

orders_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.orders_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.customers_raw")

customers_enhanced = df \
    .dropDuplicates(["customer_id"]) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("customer_city", trim(lower(col("customer_city")))) \
    .withColumn("customer_state", trim(lower(col("customer_state")))) \
    .fillna({
        "customer_city": "unknown",
        "customer_state": "unknown"
    })

customers_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.customers_enhanced")

In [0]:
from pyspark.sql.functions import col, lower, trim
df = spark.table("ecommerce_dev.bronze.products_raw")

display(df)

products_enhanced = df \
    .dropDuplicates(["product_id"]) \
    .fillna({"product_category_name": "unknown"}) \
    .withColumn("product_category_name", trim(lower(col("product_category_name")))) \
    .withColumn("product_weight_g", col("product_weight_g").cast("int")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("int")) \
    .fillna({
        "product_weight_g": 0,
        "product_length_cm": 0,
        "product_height_cm": 0,
        "product_width_cm": 0,
        "product_name_lenght" :0,
        "product_description_lenght" : 0,
        "product_photos_qty" : 0,
    }) \
    .withColumn("is_category_missing", col("product_category_name") == "unknown")

products_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.products_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.sellers_raw")

sellers_enhanced = df \
    .dropDuplicates(["seller_id"]) \
    .filter(col("seller_id").isNotNull()) \
    .withColumn("seller_city", trim(lower(col("seller_city")))) \
    .withColumn("seller_state", trim(lower(col("seller_state"))))

sellers_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.sellers_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.payments_raw")

payments_enhanced = df \
    .dropDuplicates() \
    .filter(col("order_id").isNotNull()) \
    .withColumn("payment_type", trim(lower(col("payment_type")))) \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .filter(col("payment_value") > 0)

payments_agg = payments_enhanced.groupBy("order_id") \
    .agg(_sum("payment_value").alias("total_payment_value"))

payments_agg.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.payments_enhanced")

In [0]:
from pyspark.sql.functions import col, when, lit, to_timestamp
df = spark.table("ecommerce_dev.bronze.reviews_raw")

reviews_enhanced = df \
    .filter(col("order_id").isNotNull()) \
    .dropDuplicates(["review_id"]) \
    .withColumn("review_score", col("review_score").try_cast("int")) \
    .fillna({"review_score": 0}) \
    .filter(col("review_score") > 0) \
    .fillna({"review_comment_title": "", "review_comment_message": ""}) \
    .withColumn("review_category", when(col("review_score") >= 3, "positive").otherwise("negative")) \
    # .withColumn("review_answer_timestamp", when(col("review_answer_timestamp").isNull(), to_timestamp(lit("1970-01-01 00:00:00"))).otherwise(try_cast(col("review_answer_timestamp"), "timestamp"))) \
    # .withColumn("review_creation_date", when(col("review_creation_date").isNull(), to_timestamp(lit("1970-01-01 00:00:00"))).otherwise(try_cast(col("review_creation_date"), "timestamp")))

reviews_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.reviews_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.order_items_raw")

items_enhanced = df \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .filter(col("order_id").isNotNull()) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .filter(col("price") > 0) \
    .filter(col("freight_value") >= 0) \
    .withColumn("total_item_value", col("price") + col("freight_value"))

items_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.order_items_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.geolocation_raw")

geo_enhanced = df \
    .dropDuplicates() \
    .withColumn("geolocation_city", trim(lower(col("geolocation_city")))) \
    .withColumn("geolocation_state", trim(lower(col("geolocation_state")))) \
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double")) \
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))

geo_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.geolocation_enhanced")

In [0]:
df = spark.table("ecommerce_dev.bronze.product_category_name_translation_raw")

category_enhanced = df \
    .dropDuplicates() \
    .withColumn("product_category_name", trim(lower(col("product_category_name")))) \
    .withColumn("product_category_name_english", trim(lower(col("product_category_name_english"))))


category_enhanced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_dev.silver.category_translation_enhanced")